In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. 초기 세팅
INITIAL_CAPITAL = 26800000  # 약 2680만 원
TARGET_WEIGHTS = {'BTC': 0.58, 'VOO': 0.145, 'ETH': 0.12, 'GOOGL': 0.055, 'USD': 0.10}


tickers = ['BTC-USD', 'ETH-USD', 'VOO', 'GOOGL', 'USDKRW=X']
data = yf.download(tickers, period="1y", interval="1d")['Close']
data = data.ffill().dropna() # 결측치 처리 (휴일 등)

# 원화(KRW) 환산 가격표 만들기
prices_krw = pd.DataFrame(index=data.index)
prices_krw['BTC'] = data['BTC-USD'] * data['USDKRW=X']
prices_krw['ETH'] = data['ETH-USD'] * data['USDKRW=X']
prices_krw['VOO'] = data['VOO'] * data['USDKRW=X']
prices_krw['GOOGL'] = data['GOOGL'] * data['USDKRW=X']
prices_krw['USD'] = data['USDKRW=X']

# 2. 백테스트 지갑 세팅 (1년 전 첫날 기준 매수)
first_day = prices_krw.iloc[0]
wallet = {'BTC': 0, 'VOO': 0, 'ETH': 0, 'GOOGL': 0, 'USD': 0}
avg_buy_price = {'BTC': 0, 'VOO': 0, 'ETH': 0, 'GOOGL': 0}

# 첫날 목표 비중대로 초기 자산 세팅
for asset, weight in TARGET_WEIGHTS.items():
    allocated_krw = INITIAL_CAPITAL * weight
    wallet[asset] = allocated_krw / first_day[asset]
    if asset != 'USD':
        avg_buy_price[asset] = first_day[asset]

# 3. 시뮬레이션 루프
history = []
last_rebalance_month = None  #마지막으로 리밸런싱한 달을 추적

for date, daily_prices in prices_krw.iterrows():
    # 1) 현재 총자산 평가
    current_values = {asset: wallet[asset] * daily_prices[asset] for asset in TARGET_WEIGHTS.keys()}
    total_value = sum(current_values.values())

    #매월 9일(혹은 9일 이후 첫 거래일)에만 리밸런싱 실행
    if date.day >= 9 and date.month != last_rebalance_month:
        last_rebalance_month = date.month  # 이번 달 리밸런싱 완료 처리

        # 2) 리밸런싱 로직 체크 (목표 비중과 비교)
        for asset, target_w in TARGET_WEIGHTS.items():
            if asset == 'USD': continue

            current_w = current_values[asset] / total_value
            buy_threshold = target_w * 0.8
            sell_threshold = target_w * 1.2

            # [BUY 신호] 하단 밴드 이탈 시
            if current_w <= buy_threshold:
                required_krw = total_value * (target_w - current_w)
                if current_values['USD'] >= required_krw:
                    qty_to_buy = required_krw / daily_prices[asset]

                    total_cost = (wallet[asset] * avg_buy_price[asset]) + required_krw
                    wallet[asset] += qty_to_buy
                    avg_buy_price[asset] = total_cost / wallet[asset]

                    wallet['USD'] -= required_krw / daily_prices['USD']

            # [SELL 신호] 상단 밴드 이탈 시 + Profit Guard
            elif current_w >= sell_threshold:
                if daily_prices[asset] > avg_buy_price[asset]:
                    excess_krw = total_value * (current_w - target_w)
                    qty_to_sell = excess_krw / daily_prices[asset]

                    wallet[asset] -= qty_to_sell
                    wallet['USD'] += excess_krw / daily_prices['USD']

    # 일일 총자산 기록
    history.append({'Date': date, 'Total_Value': total_value})

# 4. 결과 분석 및 시각화
result_df = pd.DataFrame(history).set_index('Date')
final_value = result_df['Total_Value'].iloc[-1]
profit_rate = ((final_value - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100

print("\n" + "="*50)
print(f"시작 자본금: {INITIAL_CAPITAL:,.0f} 원")
print(f"최종 자산금: {final_value:,.0f} 원")
print(f"누적 수익률: {profit_rate:,.2f}%")
print("="*50)


plt.figure(figsize=(12, 6))
plt.plot(result_df.index, result_df['Total_Value'], label='Portfolio Value', color='#2196F3', linewidth=2)
plt.axhline(y=INITIAL_CAPITAL, color='r', linestyle='--', label='Initial Capital')
plt.title("Quant Fund - 1 Year Backtest Result", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Total Value (KRW)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()